# Data Prep
Prepare data for analysis.

In [5]:
# imports
import pandas as pd
import json

In [3]:
# load datasets in
electricity = pd.read_csv("../data/combined_electricity_data.csv")
print(f"Electricity {electricity.columns}")

water = pd.read_csv("../data/combined_water_data.csv")
print(f"Water {water.columns}")

datacenters = pd.read_pickle("../data/datacenters.pkl")
print(f"datacenters {datacenters.columns}")

Electricity Index(['state', 'year', 'sector', 'electricity_usage'], dtype='object')
Water Index(['State', 'Year', 'Water_Usage'], dtype='object')
datacenters Index(['Name', 'State', 'Operator', 'Address', 'Year Opened', 'Source URL',
       'year', 'state_extracted', 'state_code'],
      dtype='object')


### Merge datasets on state, year


Standardize column names. 
- Electricity looks good 
- Water just needs lowercase
- datacenters needs some remapping. Year needs extracted, state code needs extracted

In [4]:
# Standardize column names
water.columns = water.columns.str.lower()

In [7]:
# datacenters state code needs parsed to state
states = pd.read_json("../data/state_map.json", orient="index", typ='series')

datacenters['state'] = datacenters['state_code'].map(states)
print(datacenters['state'][0])

Texas


In [15]:
# datacenters can be trimmed to only relevant cols
datacenters.columns
datacenters = datacenters[['Name', 'state', 'year']]

In [16]:
# merge datasets
combined = pd.merge(
    electricity,
    water,
    on=['state', 'year'],
    how='outer'
)

combined = pd.merge(
    combined, 
    datacenters,
    on=['state', 'year'],
    how='outer'
)

combined.head

<bound method NDFrame.head of          state    year       sector  electricity_usage  water_usage  \
0           AK  1990.0   Commercial          1972116.0          NaN   
1           AK  1990.0   Industrial           459282.0          NaN   
2           AK  1990.0  Residential          1661311.0          NaN   
3           AK  1991.0   Commercial          2005247.0          NaN   
4           AK  1991.0   Industrial           465878.0          NaN   
...        ...     ...          ...                ...          ...   
10097  Wyoming  2028.0          NaN                NaN          NaN   
10098  Wyoming  2028.0          NaN                NaN          NaN   
10099  Wyoming  2028.0          NaN                NaN          NaN   
10100  Wyoming  2029.0          NaN                NaN          NaN   
10101  Wyoming  2029.0          NaN                NaN          NaN   

                        Name  
0                        NaN  
1                        NaN  
2                       

In [17]:
# filter data for available time range (2000-2020)
combined_filtered = combined[(combined['year'] >= 2000) & (combined['year'] <= 2020)]

In [18]:
import pickle
# save df for access in further notebooks
combined_filtered.to_pickle("../data/merged_state_data_2000_2020.pkl")